# Mathematical Foundations of ML — Working Notebook
**Muthezhil M | RA2311054010024 | Mechanical, SRM KTR**

This notebook contains all working code for the semester project: gradient-based
optimizer comparison and Markov chain text modeling. Run each cell top to bottom.

Run this in Google Colab: File -> Upload notebook -> select this file, or drag it into
https://colab.research.google.com


## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("numpy:", np.__version__)


## 2. Loss Surfaces and Gradients

Three synthetic loss surfaces, hand-derived and verified:

- **Bowl:** f(x,y) = x^2 + y^2, grad = (2x, 2y)
- **Saddle:** f(x,y) = x^2 - y^2, grad = (2x, -2y)
- **Rosenbrock:** f(x,y) = (1-x)^2 + 100(y-x^2)^2,
  grad = (-2(1-x) - 400x(y-x^2), 200(y-x^2))


In [ ]:
def bowl(x, y):
    return x**2 + y**2

def bowl_grad(x, y):
    return np.array([2*x, 2*y])

def saddle(x, y):
    return x**2 - y**2

def saddle_grad(x, y):
    return np.array([2*x, -2*y])

def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dfdx = -2*(1 - x) - 400*x*(y - x**2)
    dfdy = 200*(y - x**2)
    return np.array([dfdx, dfdy])

SURFACES = {
    "bowl": {"f": bowl, "grad": bowl_grad, "minimum": (0.0, 0.0)},
    "saddle": {"f": saddle, "grad": saddle_grad, "minimum": None},
    "rosenbrock": {"f": rosenbrock, "grad": rosenbrock_grad, "minimum": (1.0, 1.0)},
}

# sanity checks against hand-derived values
print("Bowl grad at (3,4):", bowl_grad(3, 4), "expected [6, 8]")
print("Saddle grad at (2,3):", saddle_grad(2, 3), "expected [4, -6]")
print("Rosenbrock grad at (1,1):", rosenbrock_grad(1, 1), "expected [0, 0]")


### Visualize the three surfaces

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ("Bowl:  f = x^2 + y^2", bowl, np.linspace(-4, 4, 200), np.linspace(-4, 4, 200), (0, 0)),
    ("Saddle:  f = x^2 - y^2", saddle, np.linspace(-4, 4, 200), np.linspace(-4, 4, 200), (0, 0)),
    ("Rosenbrock:  f = (1-x)^2 + 100(y-x^2)^2", rosenbrock, np.linspace(-2, 2, 300), np.linspace(-1, 3, 300), (1, 1)),
]

for ax, (title, func, xs, ys, (mx, my)) in zip(axes, configs):
    X, Y = np.meshgrid(xs, ys)
    Z = func(X, Y)
    if "Rosen" in title:
        ax.contourf(X, Y, np.log1p(Z), levels=40, cmap="viridis", alpha=0.6)
        ax.contour(X, Y, Z, levels=40, cmap="viridis")
    else:
        ax.contourf(X, Y, Z, levels=20, cmap="viridis", alpha=0.6)
        ax.contour(X, Y, Z, levels=20, cmap="viridis")
    ax.plot(mx, my, "r*", markersize=18, label="minimum")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend()

plt.tight_layout()
plt.show()


## 3. Optimizers

Starting with plain gradient descent (SGD). Momentum, RMSProp, and Adam will be
added here in the same pattern as the project progresses.


In [ ]:
def sgd(grad_fn, start, lr=0.1, steps=50):
    pos = np.array(start, dtype=float)
    path = [pos.copy()]
    for _ in range(steps):
        g = grad_fn(pos[0], pos[1])
        pos = pos - lr * g
        path.append(pos.copy())
    return np.array(path)

# quick check against hand-calculated steps
path = sgd(bowl_grad, start=(3, 4), lr=0.1, steps=5)
for i, p in enumerate(path):
    print(f"step {i}: ({p[0]:.3f}, {p[1]:.3f})")


### Visualize SGD trajectory on the Bowl

In [ ]:
path = sgd(bowl_grad, start=(3, 4), lr=0.1, steps=40)

xs = np.linspace(-4, 5, 200)
ys = np.linspace(-4, 5, 200)
X, Y = np.meshgrid(xs, ys)
Z = bowl(X, Y)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=25, cmap="viridis", alpha=0.6)
ax.plot(path[:, 0], path[:, 1], "o-", color="red", markersize=4, linewidth=1.5, label="SGD path")
ax.plot(path[0, 0], path[0, 1], "gs", markersize=12, label="start (3,4)")
ax.plot(0, 0, "y*", markersize=18, label="minimum (0,0)")
ax.set_title("SGD on the Bowl: 40 steps, lr=0.1")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

print("Final position:", path[-1])


## Next up (to be added in upcoming sessions)
- Momentum, RMSProp, Adam optimizers
- Saddle-escape metric across all optimizers
- Markov chain text generator (order-1/2/3)
- Stationary distribution + entropy analysis
